In [7]:
import math
import pandas as pd

print("Program Rekomendasi Lokasi Usaha Berbasis GIS")

Program Rekomendasi Lokasi Usaha Berbasis GIS


In [8]:
# Data kandidat lokasi usaha
kandidat = [
    {"nama": "Lokasi A", "penduduk": 8000,
     "lat": -6.2000, "lon": 106.8166},
    {"nama": "Lokasi B", "penduduk": 12000,
     "lat": -6.2100, "lon": 106.8266},
    {"nama": "Lokasi C", "penduduk": 10000,
     "lat": -6.2050, "lon": 106.8100}
]

# Data simulasi lokasi pesaing
pesaing = [
    {"lat": -6.2010, "lon": 106.8170},
    {"lat": -6.2030, "lon": 106.8190},
    {"lat": -6.2110, "lon": 106.8270},
    {"lat": -6.2120, "lon": 106.8280},
    {"lat": -6.2090, "lon": 106.8250},
    {"lat": -6.2070, "lon": 106.8110}
]

print("Jumlah kandidat:", len(kandidat))
print("Jumlah titik pesaing:", len(pesaing))

Jumlah kandidat: 3
Jumlah titik pesaing: 6


In [9]:
def jarak_km(lat1, lon1, lat2, lon2):
    radius_bumi = 6371

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a), math.sqrt(1 - a)
    )

    return radius_bumi * c

print("Fungsi perhitungan jarak berhasil dibuat.")

Fungsi perhitungan jarak berhasil dibuat.


In [10]:
hasil = []

for lokasi in kandidat:
    jumlah_pesaing = 0

    for p in pesaing:
        jarak = jarak_km(
            lokasi["lat"], lokasi["lon"],
            p["lat"], p["lon"]
        )

        if jarak <= 1:
            jumlah_pesaing += 1

    hasil.append({
        "nama": lokasi["nama"],
        "penduduk": lokasi["penduduk"],
        "jumlah_pesaing": jumlah_pesaing
    })

df = pd.DataFrame(hasil)
display(df)

,nama,penduduk,jumlah_pesaing
0,Lokasi A,8000,3
1,Lokasi B,12000,3
2,Lokasi C,10000,2


In [11]:
maks_penduduk = df["penduduk"].max()
maks_pesaing = df["jumlah_pesaing"].max()

df["skor_penduduk"] = (
    df["penduduk"] / maks_penduduk * 100
)

if maks_pesaing == 0:
    df["skor_persaingan"] = 100
else:
    df["skor_persaingan"] = (
        1 - df["jumlah_pesaing"] / maks_pesaing
    ) * 100

df["skor_akhir"] = (
    0.7 * df["skor_penduduk"]
    + 0.3 * df["skor_persaingan"]
)

df = df.sort_values(
    by="skor_akhir", ascending=False
).reset_index(drop=True)

df.insert(0, "peringkat", range(1, len(df) + 1))

display(df.round(2))

,peringkat,nama,penduduk,jumlah_pesaing,skor_penduduk,skor_persaingan,skor_akhir
0,1,Lokasi B,12000,3,100.00,0.00,70.00
1,2,Lokasi C,10000,2,83.33,33.33,68.33
2,3,Lokasi A,8000,3,66.67,0.00,46.67


In [12]:
terbaik = df.iloc[0]

print("HASIL REKOMENDASI LOKASI USAHA")
print("==============================")
print("Lokasi terbaik:", terbaik["nama"])
print("Jumlah penduduk:", int(terbaik["penduduk"]))
print("Jumlah pesaing dalam radius 1 km:",
      int(terbaik["jumlah_pesaing"]))
print("Skor akhir:", round(terbaik["skor_akhir"], 2))

HASIL REKOMENDASI LOKASI USAHA
Lokasi terbaik: Lokasi B
Jumlah penduduk: 12000
Jumlah pesaing dalam radius 1 km: 3
Skor akhir: 70.0


In [13]:
nama_file = "hasil_rekomendasi.csv"

df.to_csv(nama_file, index=False)

print("File berhasil dibuat:", nama_file)

File berhasil dibuat: hasil_rekomendasi.csv


In [14]:
from google.colab import files

files.download("hasil_rekomendasi.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>